In [ ]:
#!/usr/bin/env python3
"""
Project 13 -- Gate 2, decoupled Metric 2: 1M-scBloodNL raw-count pseudobulk +
paired DE, per cell type, to recover a REAL non-significant background pool.
Run on Kaggle (network + no wall-clock-per-call constraints; the sandbox this
was drafted in hit a hard 45s-per-call limit and ~5MB/s effective download
throughput -- Kaggle has neither problem). Companion: docs/TASK_gate2_decoupled_metric2.md,
STATE.md entries 2026-07-15a/b.

WHY THIS SCRIPT EXISTS (don't skip -- explains the design)
------------------------------------------------------------
The published meta-analysis DE tables (de_20201106.tar.gz) are PRE-FILTERED
(filename "lfc01minpct01": only genes already clearing |logFC|>=0.1 and
pct-expressed>=0.1 are even included) -- at n~100 donors, ~100% of what's
left is Bonferroni-significant (checked directly: B cell type, 1932/1934
genes). There is no usable "non-significant background" pool in that
resource. This script processes the RAW (well, QC'd + SCTransform-corrected,
still per-cell, not summary-filtered) count matrices directly, which
preserves the FULL gene panel and produces a real p-value distribution --
recovering the background pool the matched-NN design (src/10/12 discipline)
needs.

Also resolves the naive-CD4T gap for free: `1M_cell_types.tsv`'s *fine*
`cell_type` column already has "naive CD4T" as its own native label (237,873
cells) -- no marker-based sub-clustering / Azimuth needed, contrary to the
original assumption that this would require extra work.

Cell-type groups (per the frozen Calderon mapping, STATE.md 2026-07-14/15):
    B          <- cell_type == "B"
    monocyte   <- cell_type in {mono 1, mono 2, mono 3, mono 4}       (pooled, Calderon's "Monocytes" is undivided)
    CD8T       <- cell_type in {memory CD8T, naive CD8T}              (pooled, Calderon's "CD8pos_T" is undivided)
    naive_CD4T <- cell_type == "naive CD4T"                           (native fine label, NOT pooled CD4T)
Excluded: the ambiguous "memory CD8T left and naive CD8T right" doublet-call
label, and "naive CD4T transitioning to stim" (not clean baseline naive).

Condition definition: per donor per cell type, baseline = all UT cells;
stim = ALL non-UT cells pooled (any of 3hCA/24hCA/3hPA/24hPA/3hMTB/24hMTB;
Calderon's own "S" was itself a single generic stimulated state, so pooling
here is the closest analog -- avoids arbitrarily picking one pathogen/
timepoint). This is an explicit, documented modeling choice, not hidden.

Only donors with BOTH a baseline and >=1 stim sample, per cell-type group,
are used (paired design) -- 104/120 donors qualify for all 4 groups (checked
in the sandbox before handing this off).

Pipeline
--------
1. Download small metadata (1M_cell_types.tsv, 1M_assignments_conditions_expid.tsv)
   + barcodes/features for both chemistries (v2, v3) -- all small/fast.
2. Build barcode -> (group, donor, is_stim) bucket assignment; only keep
   paired donors per group. Map to per-chemistry column-index -> bucket_id
   arrays (barcodes.tsv order = mtx column order).
3. Download the two SCT-normalized matrices (v2 ~1.8GB, v3 ~1.7GB gzipped,
   MatrixMarket coordinate format). Stream-parse in chunks (pandas
   read_csv, C engine) and accumulate sum + count per (gene_row, bucket)
   via vectorized numpy scatter-add -- never materializes the full dense
   or even full sparse matrix in memory.
4. Combine v2 + v3 accumulators by GENE SYMBOL (chemistries have different
   gene panels/order -- v2: 20,027 genes, v3: 22,050 genes, checked to be
   near-identical but not identical order, must join on symbol not index).
5. Per (group, donor): baseline_mean = sum/count over UT cells;
   stim_mean = sum/count over pooled non-UT cells. Paired t-test across
   donors (baseline_mean vs stim_mean), per gene, per group. BH-correct.
6. Output: one parquet per group with full gene-level stats (p, q, mean
   diff, baseline level) -- this IS the real background pool source the
   sandbox couldn't produce from the published summary tables.

Output
------
    /kaggle/working/scbloodnl_de_<group>.parquet   (one per of the 4 groups)
    /kaggle/working/scbloodnl_de_report.json        (run metadata, gene/donor counts)
Bring these 4 parquet files + the report back into data/processed/ in the
main repo to continue Step 3 (matching) of docs/TASK_gate2_decoupled_metric2.md.

Usage
-----
    python kaggle_1m_scbloodnl_de.py
"""
from __future__ import annotations
import gzip
import json
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

IS_KAGGLE = Path("/kaggle").exists()
WORK = Path("/kaggle/working") if IS_KAGGLE else Path(__file__).resolve().parent
RAW = WORK / "raw_1m_scbloodnl"
RAW.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://molgenis26.gcc.rug.nl/downloads/1m-scbloodnl"

GROUP_MAP = {
    "B": "B",
    "mono 1": "monocyte", "mono 2": "monocyte", "mono 3": "monocyte", "mono 4": "monocyte",
    "memory CD8T": "CD8T", "naive CD8T": "CD8T",
    "naive CD4T": "naive_CD4T",
}
GROUPS = ["B", "monocyte", "CD8T", "naive_CD4T"]

CHUNK_ROWS = 5_000_000


def fetch(fname, dest=None):
    dest = dest or (RAW / fname)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[fetch] {fname} already present ({dest.stat().st_size/1e6:.1f}MB), skip")
        return dest
    url = f"{BASE_URL}/{fname}"
    print(f"[fetch] downloading {url} ...")
    t0 = time.time()
    urllib.request.urlretrieve(url, dest)
    print(f"[fetch] done {fname}: {dest.stat().st_size/1e6:.1f}MB in {time.time()-t0:.1f}s")
    return dest


def build_bucket_map():
    """Barcode -> (group, donor, is_stim), restricted to paired donors per group."""
    ct_path = fetch("1M_cell_types.tsv")
    asn_path = fetch("1M_assignments_conditions_expid.tsv")
    ct = pd.read_csv(ct_path, sep="\t")
    asn = pd.read_csv(asn_path, sep="\t")
    meta = ct.merge(asn, on="barcode", how="inner")

    meta["group"] = meta.cell_type.map(GROUP_MAP)
    meta = meta.dropna(subset=["group", "assignment", "timepoint"]).copy()
    meta["is_stim"] = (meta.timepoint != "UT").astype(int)
    meta["donor"] = meta.assignment.astype(int)

    paired_check = meta.groupby(["group", "donor"])["is_stim"].nunique()
    valid_pairs = set(paired_check[paired_check == 2].index)
    meta["pair_key"] = list(zip(meta.group, meta.donor))
    meta = meta[meta.pair_key.isin(valid_pairs)].copy()

    buckets = sorted((meta.group + "|" + meta.donor.astype(str) + "|" + meta.is_stim.astype(str)).unique())
    bucket_id = {b: i + 1 for i, b in enumerate(buckets)}  # 1-indexed, 0 = unwanted
    meta["bucket_str"] = meta.group + "|" + meta.donor.astype(str) + "|" + meta.is_stim.astype(str)
    meta["bucket_id"] = meta.bucket_str.map(bucket_id)

    bucket_meta = pd.DataFrame(
        [(bid, *b.split("|")) for b, bid in bucket_id.items()],
        columns=["bucket_id", "group", "donor", "is_stim"]
    )
    bucket_meta["donor"] = bucket_meta.donor.astype(int)
    bucket_meta["is_stim"] = bucket_meta.is_stim.astype(int)

    print(f"[bucket] {len(buckets)} buckets, donors per group: "
          f"{bucket_meta.groupby('group').donor.nunique().to_dict()}")
    return meta.set_index("barcode")["bucket_id"], bucket_meta


def col_bucket_array(barcode_bucket, chem_barcodes_path):
    bcs = pd.read_csv(chem_barcodes_path, header=None)[0]
    arr = bcs.map(barcode_bucket).fillna(0).astype(np.int64).values  # 1-indexed col -> bucket_id (0=skip)
    print(f"[colbucket] {chem_barcodes_path.name}: {len(bcs)} cells, {int((arr>0).sum())} wanted")
    return arr


def stream_aggregate(mtx_path, col_bucket, n_genes, n_buckets):
    """Returns (sum_mat, count_mat): (n_genes, n_buckets) float64 arrays."""
    sum_mat = np.zeros((n_genes, n_buckets + 1), dtype=np.float64)  # +1: dummy col 0 for unwanted
    cnt_mat = np.zeros((n_genes, n_buckets + 1), dtype=np.int64)

    t0 = time.time()
    with gzip.open(mtx_path, "rt") as f:
        f.readline()  # %%MatrixMarket header
        dims = f.readline().split()
        decl_genes, decl_cells, decl_nnz = int(dims[0]), int(dims[1]), int(dims[2])
        print(f"[stream] {mtx_path.name}: declared {decl_genes} genes x {decl_cells} cells, {decl_nnz} nnz")

        reader = pd.read_csv(f, sep=" ", header=None, names=["gene", "cell", "val"],
                              dtype={"gene": np.int32, "cell": np.int32, "val": np.float64},
                              chunksize=CHUNK_ROWS)
        n_rows_seen = 0
        for chunk in reader:
            b = col_bucket[chunk["cell"].values - 1]  # 0-indexed lookup, 0 = unwanted
            keep = b > 0
            if keep.any():
                g = chunk["gene"].values[keep] - 1  # 0-indexed gene row
                bb = b[keep]
                v = chunk["val"].values[keep]
                np.add.at(sum_mat, (g, bb), v)
                np.add.at(cnt_mat, (g, bb), 1)
            n_rows_seen += len(chunk)
            if n_rows_seen % (CHUNK_ROWS * 5) == 0:
                print(f"  [stream] {n_rows_seen}/{decl_nnz} rows ({time.time()-t0:.0f}s)")
    print(f"[stream] {mtx_path.name} done: {n_rows_seen} rows in {time.time()-t0:.0f}s")
    return sum_mat[:, 1:], cnt_mat[:, 1:]  # drop dummy col 0


def process_chemistry(chem, barcode_bucket, n_buckets):
    features_path = fetch(f"10x_{chem}_SCT_features.tsv.gz")
    barcodes_path = fetch(f"10x_{chem}_barcodes.tsv.gz")
    matrix_path = fetch(f"10x_{chem}_SCT_matrix.mtx.gz")

    genes = pd.read_csv(features_path, header=None)[0].values
    n_genes = len(genes)
    col_bucket = col_bucket_array(barcode_bucket, barcodes_path)

    sum_mat, cnt_mat = stream_aggregate(matrix_path, col_bucket, n_genes, n_buckets)
    return genes, sum_mat, cnt_mat


def main():
    t0 = time.time()
    barcode_bucket, bucket_meta = build_bucket_map()
    n_buckets = bucket_meta.bucket_id.max()

    genes_v2, sum_v2, cnt_v2 = process_chemistry("v2", barcode_bucket, n_buckets)
    genes_v3, sum_v3, cnt_v3 = process_chemistry("v3", barcode_bucket, n_buckets)

    # combine by gene SYMBOL (chemistries have different panels/order)
    all_genes = sorted(set(genes_v2) | set(genes_v3))
    gidx = {g: i for i, g in enumerate(all_genes)}
    n_all = len(all_genes)
    sum_combined = np.zeros((n_all, n_buckets), dtype=np.float64)
    cnt_combined = np.zeros((n_all, n_buckets), dtype=np.int64)

    for genes, sum_mat, cnt_mat in [(genes_v2, sum_v2, cnt_v2), (genes_v3, sum_v3, cnt_v3)]:
        idx_map = np.array([gidx[g] for g in genes])
        np.add.at(sum_combined, idx_map, sum_mat)
        np.add.at(cnt_combined, idx_map, cnt_mat)

    print(f"[combine] {n_all} total genes (v2={len(genes_v2)}, v3={len(genes_v3)}) ({time.time()-t0:.0f}s)")

    mean_mat = np.divide(sum_combined, cnt_combined, out=np.full_like(sum_combined, np.nan), where=cnt_combined > 0)

    report = {"n_all_genes": n_all, "n_buckets": int(n_buckets), "groups": {}}

    for group in GROUPS:
        gb = bucket_meta[bucket_meta.group == group]
        donors = sorted(gb.donor.unique())
        baseline_bucket = {int(r.donor): int(r.bucket_id) for r in gb[gb.is_stim == 0].itertuples()}
        stim_bucket = {int(r.donor): int(r.bucket_id) for r in gb[gb.is_stim == 1].itertuples()}
        common_donors = sorted(set(baseline_bucket) & set(stim_bucket))

        base_cols = [baseline_bucket[d] - 1 for d in common_donors]
        stim_cols = [stim_bucket[d] - 1 for d in common_donors]
        base_vals = mean_mat[:, base_cols]  # (n_all, n_donors)
        stim_vals = mean_mat[:, stim_cols]

        valid_gene = (~np.isnan(base_vals)).sum(axis=1) >= 3
        valid_gene &= (~np.isnan(stim_vals)).sum(axis=1) >= 3

        diffs = stim_vals - base_vals
        n_donors_used = (~np.isnan(diffs)).sum(axis=1)
        mean_diff = np.nanmean(diffs, axis=1)
        sd_diff = np.nanstd(diffs, axis=1, ddof=1)

        pvals = np.ones(n_all)
        with np.errstate(invalid="ignore", divide="ignore"):
            se = sd_diff / np.sqrt(n_donors_used)
            tstat = mean_diff / se
            ok = valid_gene & (se > 0) & np.isfinite(tstat)
            pvals[ok] = 2 * stats.t.sf(np.abs(tstat[ok]), df=n_donors_used[ok] - 1)
        pvals[~np.isfinite(pvals)] = 1.0

        qvals = np.ones(n_all)
        if valid_gene.sum() > 0:
            qvals[valid_gene] = multipletests(pvals[valid_gene], method="fdr_bh")[1]

        baseline_mean_overall = np.nanmean(base_vals, axis=1)

        out = pd.DataFrame({
            "group": group, "gene": all_genes,
            "n_donors_used": n_donors_used,
            "baseline_mean": baseline_mean_overall,
            "mean_diff_stim_minus_baseline": mean_diff,
            "p": pvals, "q": qvals,
        })
        out = out[valid_gene].reset_index(drop=True)
        out_path = WORK / f"scbloodnl_de_{group}.parquet"
        out.to_parquet(out_path)

        n_sig = int((out.q < 0.05).sum())
        report["groups"][group] = {
            "n_donors_paired": len(common_donors), "n_genes_tested": int(valid_gene.sum()),
            "n_sig_q05": n_sig, "n_nonsig_q05": int(valid_gene.sum()) - n_sig,
        }
        print(f"[DE] {group}: n_donors={len(common_donors)} n_genes={int(valid_gene.sum())} "
              f"n_sig(q<0.05)={n_sig} n_nonsig={int(valid_gene.sum())-n_sig} -> {out_path}")

    report["total_runtime_s"] = time.time() - t0
    with open(WORK / "scbloodnl_de_report.json", "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n[done] total runtime {report['total_runtime_s']:.0f}s")
    print(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()